# Part 5: Efficient Data Handling through Data Parallelism

**Objective:** Explore and apply data parallelism techniques to enhance processing efficiency for large-scale banking data — essential as data volumes grow.

This notebook demonstrates the optimization techniques applied throughout the pipeline (`utils/spark_utils.py`, `spark/*.py`) and shows them running live on the dataset.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (
    SparkSession.builder
    .appName("BankDataParallelism_Notebook")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

df = spark.read.option("header","true").option("inferSchema","true").csv("../data/bank.csv")
print(f"Loaded {df.count()} rows")
print(f"Default partitions: {df.rdd.getNumPartitions()}")

### 5.1 Partitioning — splitting work across cores

Spark splits a DataFrame into partitions; each partition is processed by one executor core **simultaneously**. This is the basic unit of data parallelism.

```
Dataset: 4521 rows
  -> repartition(8)
Partition 0: ~565 rows  [CPU Core 1]
Partition 1: ~565 rows  [CPU Core 2]
...
Partition 7: ~565 rows  [CPU Core 8]
  -> parallel processing -> results merged
```

In [ ]:
def show_partition_info(df, label=""):
    n = df.rdd.getNumPartitions()
    dist = df.rdd.mapPartitionsWithIndex(lambda i, rows: [(i, sum(1 for _ in rows))]).collect()
    print(f"--- PARTITION INFO {label} ---")
    print(f"Total partitions: {n}")
    for pid, cnt in sorted(dist):
        bar = "#" * max(1, cnt // 50)
        print(f"  Part {pid:2d}: {cnt:5d} rows  {bar}")

show_partition_info(df, "(default read)")

df_repart = df.repartition(8)
show_partition_info(df_repart, "(after repartition(8))")

### 5.2 Caching / Persisting — avoiding recomputation

`cache()` is shorthand for `persist(StorageLevel.MEMORY_AND_DISK)`. Any DataFrame reused across multiple actions (e.g. used in both EDA and feature engineering) should be cached once instead of recomputed from the CSV each time.

In [ ]:
start = time.time()
df_repart.persist()
_ = df_repart.count()              # materializes the cache
first_pass = time.time() - start

start = time.time()
_ = df_repart.filter(F.col("balance") > 0).count()   # reuses cached data
second_pass = time.time() - start

print(f"First action (materializes cache): {first_pass:.3f}s")
print(f"Second action (reads from cache):  {second_pass:.3f}s")

### 5.3 Broadcast Joins — avoiding shuffles for small lookup tables

A **shuffle** (data redistribution across partitions for `groupBy`/`join`/`orderBy`) is the most expensive Spark operation: it involves serialization, disk writes, network transfer, and deserialization. Broadcasting a small lookup table avoids the shuffle entirely by sending a full copy to every executor.

In [ ]:
from pyspark.sql.functions import broadcast

# Small lookup table: job -> typical income bracket (illustrative)
job_income_lookup = spark.createDataFrame([
    ("management", "high"), ("technician", "mid"), ("blue-collar", "mid"),
    ("retired", "low"), ("student", "low"), ("admin.", "mid"),
    ("services", "mid"), ("self-employed", "high"), ("entrepreneur", "high"),
    ("unemployed", "low"), ("housemaid", "low"), ("unknown", "mid"),
], ["job", "income_bracket"])

joined = df.join(broadcast(job_income_lookup), on="job", how="left")
joined.groupBy("income_bracket").count().show()

### 5.4 Coalesce before writing — avoiding the small-files problem

`repartition()` increases parallelism for *processing*; `coalesce()` reduces partition count before a *write*, avoiding hundreds of tiny output files that hurt downstream read performance.

In [ ]:
out_path = "/tmp/bank_coalesce_demo.parquet"
df_repart.coalesce(2).write.mode("overwrite").parquet(out_path)

import subprocess
result = subprocess.run(["ls", "-la", out_path], capture_output=True, text=True)
print(result.stdout)

### 5.5 Optimizations Applied Across This Project

| Technique | What it does | Where used |
|-----------|---------------|------------|
| `spark.sql.shuffle.partitions=8` | Right-sizes shuffle for this dataset's scale | All scripts |
| Parquet + Snappy | Columnar storage, ~5x compression | `data/*.parquet` |
| ORC in Hive | Predicate pushdown, column pruning | `bank_orc` table |
| `handleInvalid='keep'` in encoders | Avoids job failure on unseen categories | Feature pipeline |
| `parallelism=2` in CrossValidator | Runs 2 CV folds simultaneously | Model tuning |
| `persist()` before multi-pass ops | Avoids recomputing the same DataFrame | EDA + training |
| `coalesce()` before writes | Avoids many tiny output files | Parquet writes |
| Broadcast joins | Avoids shuffles for small lookup tables | Demonstrated above |

In [ ]:
spark.stop()
print("Data parallelism & optimization demo complete")

This completes all 5 objectives: (1) Hadoop/Hive data management, (2) Spark EDA, (3) Spark ML predictive modeling, (4) Spark Streaming with window operations, (5) data parallelism & optimization.